# Fetch Raw Race Data (pynascar)
One-time script used to pull the raw Michigan Cup Series race data with [pynascar](https://github.com/ab5525/pynascar) and save it to the `data/` folder.

**You don't need to run this** to use the main notebook — the CSV files it produces are already included in this repository. It's here to show how the data was collected, and to make it easy to pull additional races later.

Requires: `pip install pynascar`

In [ ]:
import os
import pandas as pd
from pynascar import Schedule, Race

In [ ]:
# Michigan Cup Series races: {season: race_id}
races = {
    2021: 5031,
    2022: 5170,
    2023: 5296,
    2024: 5415,
    2025: 5573,
    2026: 5612,}

In [ ]:
# Pull raw tables for each race and save one combined CSV per table
################################################
os.makedirs('data', exist_ok=True)

lap_times, pit_stops, cautions, metadata = [], [], [], [] # Store Datasets

for year, race_id in races.items():
    race = Race(year, 1, race_id, reload=True)   # 1 = Cup Series

    # Raw tables, tagged with race_id so races can be separated later
    lap_times.append(race.telemetry.lap_times.assign(race_id=race_id)) 
    pit_stops.append(race.telemetry.pit_stops.assign(race_id=race_id))
    cautions.append(race.results.cautions.assign(race_id=race_id))

    # This race's row from the season schedule (scheduled laps, stage lengths, etc.)
    schedule = Schedule(year, 1).data
    metadata.append(schedule.loc[schedule['race_id'] == race_id])

pd.concat(lap_times).to_csv('data/michigan_lap_times.csv', index=False)
pd.concat(pit_stops).to_csv('data/michigan_pit_stops.csv', index=False)
pd.concat(cautions).to_csv('data/michigan_cautions.csv', index=False)
pd.concat(metadata).to_csv('data/michigan_race_metadata.csv', index=False)

print("Saved 4 Michigan files to /data")